# 2a. Workflows: add a step to a working workflow

## Learning objectives

- Start from a working workflow, and read what each step reads and writes.
- Add a step written like the others, and let the tool run only what's new or stale.
- Build the project's figure with the workflow, so the README and the numbers come from the same data in one run.

## The problem

- The workshop's exercises use one project, [a study of Palmer penguins](../../project.md), and add a capability in each exercise.
- Exercise 1 declared and locked the software the project uses. This exercise automates running it.
- `project` has a working Snakefile with three rules, `clean`, `validate` and `report`. They build `metrics.json` from the 2007 field season: how much heavier a male penguin is than a female of the same species, in grams.
- Its README shows a figure, `figures/body_mass.svg`. `plot.py` draws it, and nothing runs `plot.py`.
- A workflow like this often comes from a colleague, a lab or a public catalog. You add a step to it.

## What you'll do

1. Step 1: run the workflow you were given, and compare its numbers with this page.
2. Step 2: add a `plot` rule that draws the figure, and commit the Snakefile.
3. Step 3: change a parameter, and see which rules rerun.
4. Run the check.

- The exercise takes about 15 to 20 minutes.
- Only reading? Each step shows the output and the fix.

- [2b. Nextflow](nextflow.md) is the same exercise in Nextflow.

## The rule

- Your research products (papers, code, data) are the pets. Everything that runs them is cattle.
- Here the pets are the data, the scripts, `params.yaml`, the Snakefile and the README, all in Git.
- Everything built is cattle: `results/`, `metrics.json`, `figures/` and `.snakemake/`. They're in `.gitignore`. Delete them, and one command builds them again.
- Publishing is the site's job. This site's [results page](../../results/results.ipynb) is rebuilt from the data on every push.

## Get a terminal

## In Binder

- You read this page and paste commands into a terminal. Nothing runs on the page.
- The output shown here is recorded from the untouched project, so the check fails until you add the step. The fixes are yours.

**[Open it in Binder](https://mybinder.org/v2/gh/ohpcsim/reproducibility-in-ai/binder-02-workflows?urlpath=lab/workspaces/exercise)**

- Free, with nothing to install. The first start can take a few minutes.

- The terminal is on the left, already in the `project` folder. A read-only copy of this page is on the right.
- Copy and paste a command: Ctrl+C, then Ctrl+V in the terminal. On a Mac, Cmd+C and Cmd+V.
- A session is temporary and has up to 2 GB of memory. [Download anything you want to keep](../../setup.md#download-your-work).


**Editing files**

- In Binder, double-click a file in the file panel on the left to open it in JupyterLab's editor. Save with Ctrl+S, or Cmd+S on a Mac.
- Or edit it in the terminal with `nano <file>`, `vi <file>` or `vim <file>`. Binder has all three.
- nano is the simplest. Ctrl+O, then Enter, saves. Ctrl+X quits.

- On Windows, keep the folder in your WSL2 home, not under `/mnt/c`.

- You need [conda](../../setup.md). You don't need Jupyter.
- Paste this once:

```bash
git clone --branch binder-02-workflows --single-branch https://github.com/ohpcsim/reproducibility-in-ai.git workflows
cd workflows
conda env create --quiet --file .binder/environment.yml
conda activate repro-workflows
python scripts/setup_project.py
cd project
```

- If `conda env create` says the prefix already exists, an environment with that name is left from an earlier run. The rest of the block then uses that old environment and leaves a half-built `workflows` folder. Remove both, then paste the block again: `conda deactivate; conda env remove -n repro-workflows -y; cd ../..; rm -rf workflows`
- The block leaves you in `workflows/project`, so `cd ../..` goes to the folder above `workflows`. If `cd project` failed too, use `cd ..` instead.
- Every command below runs in that `project` folder.
- In a new terminal, turn conda on as in [Set up](../../setup.md), then run `conda activate repro-workflows` and `cd workflows/project` from the folder above `workflows`.
- To start over, run `cd ..` to leave `project`, then `python scripts/setup_project.py --reset`. This deletes your work.

**Editing files**

- Use any text editor.
- If you're not sure, `nano FILE` works in the terminal. Ctrl+O, then Enter, saves. Ctrl+X quits.

**With uv instead of conda**

- The terminal starts in the exercise's conda environment, and `project` is already built. To use uv instead, build `.venv` from the exercise's `uv.lock` in the folder above `project`, and activate it:

```bash
cd ~
uv sync --locked
source .venv/bin/activate
cd project
```

- You don't run `scripts/setup_project.py`. Binder ran it when it built the session, and a second run leaves `project` as it is.
- `uv sync` downloads Python 3.12.12 and the locked packages. It took 7 to 11 seconds on mybinder.org.
- A new terminal starts in conda's environment again. Run `source ~/.venv/bin/activate` in it.

**With uv instead of conda**

- [Install uv](../../setup.md#install-uv) first. Then paste this instead of the block above:

```bash
git clone --branch binder-02-workflows --single-branch https://github.com/ohpcsim/reproducibility-in-ai.git workflows
cd workflows
uv sync --locked
source .venv/bin/activate
uv run python scripts/setup_project.py
cd project
```

- `uv sync --locked` downloads Python 3.12.12 itself. It also builds connection-pool, which Snakemake needs, from source. That package is pure Python, so it needs no compiler.
- Activation puts Python and Snakemake on your path. The page's own commands then work as written.
- If you reopen a terminal, go to the exercise folder, run `source .venv/bin/activate`, then `cd project`.

## The project

- `project` has the 2007 penguins, four scripts, `params.yaml`, a README and a Snakefile, in one commit. Nothing built is in Git.

```bash
cat README.md
cat Snakefile
cat .gitignore
git log --format='%h  %s'
```

In [1]:
# Recorded output. You don't need to run this.
import sys

sys.path.insert(0, ".exercise")
from page import reset, run

run("cat README.md", "cat Snakefile", "cat .gitignore", "git log --format='%h  %s'")


$ cat README.md
# Sexual dimorphism in Palmer penguins

How much heavier is a male penguin than a female of the same species? We report the difference of the mean body masses, in grams, per species, for the 2007 field season.

![Body mass by species and sex](figures/body_mass.svg)

## Rebuild the results

```bash
snakemake --cores 1
```

- The workflow reads `data/penguins.csv` and writes `metrics.json`, the difference per species, and `figures/body_mass.svg`, the figure above. Neither is in Git. The workflow builds them.
- `params.yaml` sets the statistic: `mean` or `median`.
- `clean.py` drops penguins with no body mass or no sex. `validate.py` checks the columns and rows. `report.py` writes `metrics.json`. `plot.py` draws the figure.

Data: palmerpenguins (Horst, Hill and Gorman 2020, doi:10.5281/zenodo.3960218), CC0, from Gorman, Williams and Fraser (2014), PLOS ONE 9(3): e90081. Collected by Palmer Station LTER with NSF support.

$ cat Snakefile
configfile: "params.yaml"

rule all

## Step 1: run the workflow you were given

**Problem:** before you change a workflow, run it, and check that you get the numbers its authors got.

```bash
snakemake -n
snakemake --cores 1
cat metrics.json
git status --short
```

In [2]:
# Recorded output. You don't need to run this.
run("snakemake -n", "snakemake --cores 1", "cat metrics.json", "git status --short")


$ snakemake -n
Using workflow specific profile profiles/default for setting default command line arguments.
Assuming unrestricted shared filesystem usage.
Building DAG of jobs...
Job stats:
job         count
--------  -------
clean           1
validate        1
report          1
all             1
total           4

Job stats:
job         count
--------  -------
clean           1
validate        1
report          1
all             1
total           4

Reasons:
    (check individual jobs above for details)
    input files updated by another job:
        all, report, validate
    output files have to be generated:
        clean, report, validate
This was a dry-run (flag -n). The order of jobs does not reflect the order of execution.

$ snakemake --cores 1
Using workflow specific profile profiles/default for setting default command line arguments.
Assuming unrestricted shared filesystem usage.
Building DAG of jobs...
Using shell: /usr/bin/bash
Provided cores: 1 (use --cores to define paral

**Facts**

- A Snakefile is a declaration. Each rule names its input files, its output files and the command. Snakemake builds the graph of steps from the file names. It reruns a rule when an output is missing, an input is newer than its output, or the rule's parameters changed.
- The first rule is the default target. `rule all` lists the final files, here only `metrics.json`.
- `configfile: "params.yaml"` loads the parameters into `config`. `report`'s `params:` line names the section `report.py` reads, so a change to it reruns `report`.
- `snakemake -n` is a dry run. It lists what would run and runs nothing. `snakemake --cores 1` runs it.
- The project's `profiles/default/config.yaml` sets the greedy scheduler, which is fine for a small workflow and avoids a solver that isn't available on every machine. It also leaves out the 17-line SNAKEMAKE banner and each job's details, so you see the job table and what the scripts print. Because of the profile, Snakemake prints `Using workflow specific profile profiles/default` first. `--quiet host` on the command line brings back each job's inputs, outputs and the reason it runs. Without the profile, the banner is normal and harmless.
- `metrics.json` holds the numbers on this page: Adelie males are 648.9 g heavier, Chinstrap males 250.0 g and Gentoo males 934.2 g, from 103 penguins. The same numbers from the raw data, on your machine, show that the workflow reproduces the result.
- `git status --short` prints nothing. The outputs are in `.gitignore`.

**Steps**

1. Run the commands above, and check that `metrics.json` has the numbers on this page.
2. Read the Snakefile. For each rule, find what it reads and what it writes. `validate` reads `results/clean.csv`, which `clean` writes, and writes `results/validated.csv`, which `report` reads.

## Step 2: add a step

**Problem:** the README shows a figure that nothing builds.

```bash
grep body_mass README.md
ls figures
```

In [3]:
# Recorded output. You don't need to run this.
run("grep body_mass README.md", "ls figures")


$ grep body_mass README.md
![Body mass by species and sex](figures/body_mass.svg)
- The workflow reads `data/penguins.csv` and writes `metrics.json`, the difference per species, and `figures/body_mass.svg`, the figure above. Neither is in Git. The workflow builds them.

$ ls figures
ls: cannot access 'figures': No such file or directory


- On a Mac, `ls` says `ls: figures: No such file or directory`. Both messages mean the folder isn't there yet.

**Facts**

- `plot.py` reads `results/validated.csv` and writes `figures/body_mass.svg`, body mass by species and sex.
- The first build in a new environment prints `Matplotlib is building the font cache; this may take a moment.` Then `plot` takes about 12 seconds, once. Later builds reuse the cache.
- A new step is a rule written like the others: its inputs, its output and its command. The script is an input too, so an edit to `plot.py` reruns the rule.
- `rule all` asks for the final files. Add the figure to it, or Snakemake won't build it by default.
- Snakemake runs only what's missing or stale. The other rules' outputs are current, so the dry run lists `plot` and `all` and nothing else.
- The figure is built like `metrics.json`, and it's in `.gitignore` like `metrics.json`. You commit the rule that draws it.
- `plot.py` draws the same bytes from the same data in the same environment. The SVG has no date, and its IDs are fixed. A different font library moves the text by a fraction of a point, so the check reads what the figure plots, not its bytes.

**Steps**

1. Open the Snakefile, and add a rule named `plot`, written like `report`. It reads `results/validated.csv` and `plot.py`, writes `figures/body_mass.svg`, and runs `python plot.py`. Indent with four spaces:

```text
rule plot:
    input: "results/validated.csv", "plot.py"
    output: "figures/body_mass.svg"
    shell: "python plot.py"
```

2. Add the figure to `rule all`:

```text
rule all:
    input: "metrics.json", "figures/body_mass.svg"
```

3. Ask what would run. If Snakemake reports an error instead, read the line number it names.

```bash
snakemake -n
```

- The job table lists `plot` and `all`. `clean`, `validate` and `report` are current:

```text
Job stats:
job      count
-----  -------
plot         1
all          1
total        2
```

4. Run it, and look at the figure.

```bash
snakemake --cores 1
ls figures
```

- In Binder, right-click `README.md` in the file panel, choose Open With, then Markdown Preview. The figure is in the README now.

- Open `figures/body_mass.svg` in a web browser.

- Copy `figures/body_mass.svg` to your laptop with `scp` and open it there.

5. Commit the Snakefile. `git status --short` prints nothing after the commit, because the figure is built, not committed.

```bash
git add Snakefile
git commit -m "Draw the figure with a plot rule"
git status --short
```

## Step 3: change a parameter

**Problem:** `params.yaml` says `statistic: mean`. Change it to `median`, and which rules have to rerun? You ask Snakemake.

**Facts**

- Only `report` declares `config["report"]`, so a change to `statistic` reruns `report`. `plot` reads `results/validated.csv`, which doesn't change, so the figure stays as it is.
- Had you listed `params.yaml` as an input instead, an edit to any part of the file would rerun the rule, even a part its script doesn't read. Declare what each step reads, no more.
- A decision such as which statistic to report lives in `params.yaml`, in Git, not in a command somebody typed.

**Steps**

1. Open `params.yaml`, change `statistic: mean` to `statistic: median`, and save.
2. Ask what would run, then run it, and read the new numbers.

```bash
snakemake -n
snakemake --cores 1
cat metrics.json
```

- The dry run's job table lists `report` and `all`, not `plot`. Under Reasons it says `params have changed since last execution: report`. The new numbers are the medians:

```text
{
  "dimorphism_g_adelie": 625.0,
  "dimorphism_g_chinstrap": 200.0,
  "dimorphism_g_gentoo": 925.0,
  "n_rows": 103
}
```

3. Change it back to `statistic: mean`, save, and run again. `report` reruns once more, `metrics.json` has the numbers from step 1 again, and `git status --short` prints nothing: `params.yaml` is as you committed it.

```bash
snakemake --cores 1
cat metrics.json
git status --short
```

## Check your work

- Run this when all three steps are done. Each line says what to do next.
- It clones your last commit into a throwaway folder and builds everything from the raw data with your Snakefile. It compares the numbers with this page, and checks which penguins the figure plots. Then it changes `report.statistic`, and checks that `report` reruns and `clean` and `plot` don't.
- It takes a few seconds.

```bash
python ../check.py
```

On the starter project it fails like this:

In [4]:
# Recorded output. You don't need to run this.
reset()
run("python ../check.py")


$ python ../check.py
FAIL  Snakemake has no plot rule to run for the default target. Add rule plot, and add figures/body_mass.svg to the input of rule all (step 2). Then commit the Snakefile.
PASS  In a clean clone, snakemake --cores 1 builds metrics.json from the raw data, and the numbers match the page
FAIL  The build wrote no figures/body_mass.svg. README.md shows it, so declare the step that draws it (step 2).
PASS  A changed report.statistic reruns report, not clean or plot

2 of 4 parts pass.


## Summary

- Start from a working workflow. Run it, compare its result with a published number, and read what each rule reads and writes.
- Add a step written like the others, and add its output to `rule all`. Snakemake runs only what's new or stale.
- The README's figure is a workflow output, built from the same data as the numbers. Git holds the data, the scripts and the workflow, and one command builds the rest.
- Declare the parameters a step reads, so a change reruns that step and no other.
- Exercise 3 runs the same steps as DVC stages, and versions the data and everything built from it.

- Nextflow does what Snakemake does. [2b. Nextflow](nextflow.md) is this exercise with a `main.nf`.

## On your own project

- Start from the workflow you have: yours, your lab's, or one from the [Snakemake workflow catalog](https://snakemake.github.io/snakemake-workflow-catalog/). Run it once, and compare its result with a number you trust.
- Add your step as a rule like the others. Every file it reads is an input, and every parameter it reads goes in its params.
- Have the workflow draw the figures your paper or README shows, and put them in `.gitignore`. Commit the code that draws them.

## About the data

- `data/penguins.csv` is the 2007 field season of the Palmer penguins, 110 penguins, unchanged from the [palmerpenguins](https://allisonhorst.github.io/palmerpenguins/) package. 103 of them have a body mass and a sex.
- Data: palmerpenguins (Horst, Hill and Gorman 2020, doi:10.5281/zenodo.3960218), CC0, from Gorman, Williams and Fraser (2014), PLOS ONE 9(3): e90081. Collected by Palmer Station LTER with NSF support.

## When you're done

From the folder above `workflows`:

```bash
conda deactivate
conda env remove --yes --name repro-workflows
rm -rf workflows
```

- With uv, run `deactivate` instead of the two conda lines. `rm -rf workflows` removes `.venv` too.